<a href="https://colab.research.google.com/github/matsunagalab/lecture_ML/blob/main/machine_learning_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第12回 事後学習: ニューラルネットワークその2 — ニューラルネットの学習

この notebook は講義の**事後学習用**です。目安時間は **45分** 程度。

> 前回 (第11回) は、ニューラルネットが「**線形モデル + 活性化関数**の積み重ね」でできた複雑な**関数**であることを見ました。今回のテーマは、その**パラメータ (重み $W$・バイアス $b$) をデータからどう決める (=学習する) か**です。キーワードは **勾配降下法** と **誤差逆伝播法**。スライドの図を、手を動かしながら確かめていきます。

## 到達目標
1. ニューラルネットの「**学習**」とは、予測の悪さを測る**損失関数 $L$ を最小化**するパラメータを探すことだと理解する
2. **勾配降下法** $W \leftarrow W - \eta\,\nabla L(W)$ を**自分で実装**し、損失が下がっていく様子を可視化する
3. **学習率 $\eta$** の大きさが収束を左右する (小さすぎ→遅い、大きすぎ→発散) ことを体感する
4. **誤差逆伝播法**が「合成関数の微分 (連鎖律) を出力側から計算する効率的な方法」であることを理解し、PyTorch の**自動微分** (`loss.backward()`) が勾配を計算していることを確認する
5. PyTorch でニューラルネットを学習させ、**曲線の回帰**と**線形分離できないデータの識別**ができることを確かめる

## 進め方
- 上から順にセルを実行してください。
- この回のモデルは小さいので **GPU は必須ではありません** (CPU でも十分速いです)。次回 (第13回) で本格的なネットを扱うので、Colab の「**ランタイム → ランタイムのタイプを変更 → T4 GPU**」を選んでおいてもよいです。
- 「**勾配降下法 (手で実装) → 学習率 → 誤差逆伝播 (自動微分) → PyTorch で回帰 → 識別 → 高度な話題**」という流れです。

---
# 0. 準備

PyTorch と描画ライブラリ、データ生成用の scikit-learn を読み込みます。

In [ ]:
import numpy as np
import torch
from torch import nn, optim
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# GPU (Colab の T4 など) があれば自動で使う。なければ CPU で動く (この回は CPU で十分)。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version:", torch.__version__, "| device:", device)

---
# 1. 「学習」とは損失関数を最小化すること

モデルの予測 $\hat{y}$ が、正解 $y$ とどれくらいズレているかを測るのが**損失関数** $L(W)$ です (例: 回帰なら二乗誤差 $\sum_n (y_n - \hat{y}_n)^2$)。
**学習とは、この $L(W)$ を最小にするパラメータ $W$ を探すこと**です。

第3回で習った**線形モデル**は、$L$ を $W$ で微分して $=0$ と置くと**解析解**(式で書ける答え)が得られました:

$$
\hat{w}_1 = \frac{\sum_n (x_n - \bar{x})(y_n - \bar{y})}{\sum_n (x_n - \bar{x})^2}, \qquad \hat{w}_0 = \bar{y} - \hat{w}_1 \bar{x}
$$

ところが**ニューラルネットは非線形**(合成関数のオバケ)なので、微分して $=0$ と置いても**解析的には解けません**。そこで、コンピュータで**少しずつ $W$ を動かして $L$ を下げていく**数値計算を行います:

$$
W \leftarrow W + \Delta W
$$

問題は「$\Delta W$ をどう選べば $L$ が**順調に**下がるか」です。あてずっぽうではうまくいきません。その答えが次の**勾配降下法**です。

---
# 2. 勾配降下法を手を動かして実装する

**勾配 (gradient)** $\nabla L(W)$ は、$L$ が**最も大きく増える方向**を指すベクトルです:

$$
\nabla L(W) = \left[\frac{\partial L}{\partial w_1},\ \cdots,\ \frac{\partial L}{\partial w_M}\right]^{\top}
$$

損失を**減らしたい**のだから、勾配の**逆方向**へ進めばよい。これが**勾配降下法**です:

$$
W \leftarrow W - \eta\,\nabla L(W)
$$

$\eta$ (イータ) は**学習率**で、1 歩の大きさを決めます。

スライドと同じ例題 $L(w_1, w_2) = \tfrac{1}{2}(w_1^2 + w_2^2)$ で試しましょう。これは原点を底とする「お椀」型で、勾配は手で計算できます:

$$
\frac{\partial L}{\partial w_1} = w_1, \quad \frac{\partial L}{\partial w_2} = w_2
\quad\Longrightarrow\quad \nabla L(W) = [w_1,\ w_2]^{\top}
$$

ある初期値から始めて、更新式を何度も回し、$W$ がたどった軌跡を等高線の上に描きます。

In [ ]:
def L(w):            # 損失関数 L(w1,w2) = 0.5*(w1^2 + w2^2)
    return 0.5 * (w**2).sum()

def grad_L(w):       # 勾配 ∇L = [w1, w2]  (手で求めた式)
    return w.copy()

# 勾配降下法
w = np.array([-3.0, 4.0])     # 初期値
eta = 0.2                     # 学習率
path = [w.copy()]
for step in range(40):
    w = w - eta * grad_L(w)   # W <- W - η ∇L(W)
    path.append(w.copy())
path = np.array(path)
print("最終的な w:", w, " (最小値である原点 [0,0] に近づくはず)")

# 等高線 + 降下の軌跡を描く
gx, gy = np.meshgrid(np.linspace(-4.5, 4.5, 200), np.linspace(-4.5, 4.5, 200))
Lgrid = 0.5 * (gx**2 + gy**2)
fig, ax = plt.subplots(figsize=(6, 6))
ax.contour(gx, gy, Lgrid, levels=15, cmap="gray", linewidths=0.8)
ax.plot(path[:, 0], path[:, 1], "o-", color="tab:red", ms=4, lw=1.2, label="勾配降下の軌跡")
ax.scatter([0], [0], marker="*", s=250, color="tab:blue", zorder=5, label="最小値")
ax.set_xlabel("w1"); ax.set_ylabel("w2"); ax.set_aspect("equal"); ax.legend()
plt.show()

**観察**: 軌跡は**等高線に直交する向き**(=最も急に下る向き)に、お椀の底 (最小値の原点) へ向かって降りていきます。これが勾配降下法のエッセンスです。ニューラルネットの学習も、形は複雑でも、やっていることはこの「坂を下る」操作の繰り返しです。

## 学習率 $\eta$ の影響

1 歩の大きさ $\eta$ は学習の成否を大きく左右します。同じ初期値・同じ損失関数で、$\eta$ を変えて**損失の下がり方**を比べてみましょう。

In [ ]:
def run_gd(eta, steps=30, w0=(-3.0, 4.0)):
    """学習率 eta で勾配降下し、各ステップの損失 L を記録して返す。"""
    w = np.array(w0)
    losses = []
    for _ in range(steps):
        losses.append(float(L(w)))
        w = w - eta * grad_L(w)
    losses.append(float(L(w)))
    return losses

fig, ax = plt.subplots(figsize=(7, 5))
for eta, label in [(0.05, "η=0.05 (小さすぎ: 遅い)"),
                   (0.5,  "η=0.5  (適切: スムーズに減少)"),
                   (2.05, "η=2.05 (大きすぎ: 発散)")]:
    ax.plot(run_gd(eta), "o-", ms=3, label=label)
ax.set_xlabel("更新回数 (ステップ)"); ax.set_ylabel("損失 L")
ax.set_yscale("log")                 # 発散と収束を同時に見るため対数軸
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**観察** (縦軸は対数):
- **小さすぎる** $\eta$: 1 歩が小さく、なかなか底に着かない (学習が遅い)。
- **適切な** $\eta$: 損失がスムーズに下がり、すばやく最小値へ。
- **大きすぎる** $\eta$: 1 歩で行き過ぎて**かえって損失が増え、発散**してしまう。

「学習がうまくいかない」ときは、まず学習率を疑うのが定石です。

---
# 3. 誤差逆伝播法と自動微分

勾配降下法には**勾配 $\nabla L$** が必要です。さきほどは $\nabla L = [w_1, w_2]$ と手で求められましたが、ニューラルネットは**何層もの入れ子 (合成関数)** なので、$\dfrac{\partial L}{\partial w}$ を求めるには**合成関数の微分 (連鎖律)** を何度も適用する必要があり、一見とても複雑です:

$$
\frac{\partial L}{\partial w^{(l)}_{ji}} = \frac{\partial L}{\partial u^{(l)}_{j}}\,\frac{\partial u^{(l)}_{j}}{\partial w^{(l)}_{ji}}, \qquad \frac{dy}{dx} = \frac{dy}{dy}\frac{dy}{db}\frac{db}{da}\frac{da}{dx}
$$

ここで効くのが**誤差逆伝播法 (backpropagation)** です。連鎖律の積は**どの順で計算してもよい**ので、**出力側から入力側へ**順に微分を伝えていくと、**一度の伝播であらゆる層の勾配**がまとめて得られます。これが「逆伝播」の正体です。

PyTorch をはじめとする深層学習フレームワークは、この逆伝播を**自動微分 (autograd)** として内蔵しています。`loss.backward()` と書くだけで勾配が計算されます。本当に手計算と一致するか、さきほどの $L = \tfrac12(w_1^2+w_2^2)$ で確かめましょう。

In [ ]:
# w を「勾配を追跡する」テンソルとして定義する
w = torch.tensor([-3.0, 4.0], requires_grad=True)

Lval = 0.5 * (w**2).sum()   # 順伝播: 損失を計算
Lval.backward()             # 逆伝播 (誤差逆伝播法): 勾配を自動計算

print("autograd が求めた勾配 :", w.grad.tolist())
print("手で求めた勾配 [w1,w2]:", [w[0].item(), w[1].item()])

**観察**: 自動微分の結果 `w.grad` が、手で求めた $[w_1, w_2]$ と**ぴったり一致**します。$L=\tfrac12(w_1^2+w_2^2)$ では簡単すぎてありがたみが薄いですが、**何万個ものパラメータを持つニューラルネットでも、`backward()` 一発で全パラメータの勾配が得られる**のが誤差逆伝播法の威力です。次の節からは、この仕組みに乗っかって実際のニューラルネットを学習させます。

---
# 4. PyTorch でニューラルネットを学習させる ① 曲線の回帰

ここからは PyTorch にネットワーク・損失・勾配計算を任せます。学習ループの中身は、これまで見てきた

1. **順伝播**: 予測を計算する (`model(X)`)
2. **損失**を計算する (`loss_fn(pred, Y)`)
3. **逆伝播**で勾配を求める (`loss.backward()`)
4. 勾配降下法で**パラメータを更新**する (`optimizer.step()`)

の繰り返しです。`optimizer` には、勾配降下法を改良した **Adam** を使います。

例として、3 次関数 $y = x^3$ に従うデータ (ノイズあり) を、活性化関数 **ReLU** を挟んだニューラルネットで回帰します。

In [ ]:
# データ生成: y = x^3 + ノイズ
torch.manual_seed(123); np.random.seed(123)
N = 100
X = 2.0 * (np.random.rand(N) - 0.5)
Y = X**3 + 0.01 * np.random.randn(N)
X = torch.from_numpy(X).float().view(N, 1).to(device)
Y = torch.from_numpy(Y).float().view(N, 1).to(device)

# モデル: 線形層 + ReLU を積み重ねた多層パーセプトロン (スライドと同じ構成)
model = nn.Sequential(
    nn.Linear(1, 10),  nn.ReLU(),   # 入力層
    nn.Linear(10, 10), nn.ReLU(),   # 中間層
    nn.Linear(10, 10), nn.ReLU(),   # 中間層
    nn.Linear(10, 1),               # 出力層
).to(device)

loss_fn   = nn.MSELoss()                       # 損失: 平均二乗誤差 (MSE)
optimizer = optim.Adam(model.parameters())     # 最適化: Adam (勾配降下法の発展版)

# 学習ループ
EPOCHS = 5000
history = []
for epoch in range(EPOCHS):
    pred = model(X)              # 1. 順伝播
    loss = loss_fn(pred, Y)      # 2. 損失
    optimizer.zero_grad()        #    勾配をリセット
    loss.backward()             # 3. 逆伝播 (誤差逆伝播法で勾配を計算)
    optimizer.step()             # 4. パラメータ更新 (W <- W - η∇L)
    history.append(loss.item())
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss {loss.item():.5f}")

# 結果を描画
X_test = torch.linspace(-1, 1, 100).view(100, 1).to(device)
with torch.no_grad():
    Y_test = model(X_test)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.plot(history); a1.set_xlabel("エポック"); a1.set_ylabel("損失 L")
a1.set_yscale("log"); a1.set_title("損失の減少"); a1.grid(alpha=0.3)
a2.scatter(X.cpu(), Y.cpu(), s=15, label="データ")
a2.plot(X_test.cpu(), Y_test.cpu(), "r", lw=2, label="モデル")
a2.set_xlabel("X"); a2.set_ylabel("Y"); a2.set_title("回帰の結果"); a2.legend()
plt.show()

**観察**: 左図のように**損失が単調に下がり** (スライドの「損失関数の減少の様子」そのもの)、右図ではニューラルネットの出力 (赤線) が 3 次関数のデータをうまく追えています。勾配降下法 + 誤差逆伝播法という今日の道具立てで、非線形な関数が学習できました。

---
# 5. PyTorch でニューラルネットを学習させる ② 線形分離できないデータの識別

次は**識別 (分類)** です。内側のかたまりと外側のリングからなる「**ドーナツ型**」データは、直線 1 本では分けられません (線形分離不可能)。ニューラルネットなら**閉じた曲線**の決定境界を作れるはずです。

データは第11回でも使った、内外がはっきり分かれた**きれいなドーナツ**を使います (`make_circles` の `factor` と `noise` を調整したもの)。

In [ ]:
from sklearn.datasets import make_circles

# きれいなドーナツ型データ (内側の円と外側のリングがはっきり分かれている)
X_donut, y_donut = make_circles(n_samples=800, factor=0.35, noise=0.08, random_state=0)
X_donut = (X_donut * 6).astype(np.float32)

Xt = torch.tensor(X_donut, device=device)
yt = torch.tensor(y_donut, dtype=torch.long, device=device)

# モデル: 中間層 1 つ (幅16) + 出力2クラス
net = nn.Sequential(
    nn.Linear(2, 16), nn.Tanh(),
    nn.Linear(16, 2),
).to(device)
criterion = nn.CrossEntropyLoss()                  # 識別用の損失
optimizer = optim.Adam(net.parameters(), lr=0.03)

for epoch in range(400):
    out = net(Xt)
    loss = criterion(out, yt)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss {loss.item():.4f}")

# 決定境界を描く (平面全体を分類して色分け)
xmin, xmax = X_donut[:, 0].min() - 1, X_donut[:, 0].max() + 1
ymin, ymax = X_donut[:, 1].min() - 1, X_donut[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(xmin, xmax, 300), np.linspace(ymin, ymax, 300))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=device)
with torch.no_grad():
    Z = net(grid).argmax(1).cpu().numpy().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")          # 決定境界 (領域の色)
ax.scatter(X_donut[:, 0], X_donut[:, 1], c=y_donut, cmap="coolwarm",
           s=12, edgecolor="k", linewidth=0.2)              # データ点
ax.set_aspect("equal"); ax.set_title("ニューラルネットによる識別 (決定境界)")
plt.show()

**観察**: 決定境界が**閉じたリング状**になり、内側のクラスと外側のクラスをきれいに分離しています。直線では絶対に作れない境界を、活性化関数つきのニューラルネットが学習で獲得しました。第11回で見た「ニューラルネットは空間を分離しやすい形に変形する」という話と、今日の「その変形を勾配降下法で学習する」という話がつながっています。

---
# 6. (高度な話題) 勾配降下法は局所解にトラップされる

お椀型の損失と違い、本物の損失関数は**山あり谷あり**で、谷 (極小値) がいくつもあります。勾配降下法は「いま居る場所から下る」だけなので、出発点によっては**いちばん深い谷 (大域解) ではない浅い谷 (局所解) で止まってしまう**ことがあります。

1 次元の損失 $L(x) = \sin(3x) + 0.3x^2$ で、**出発点を変える**と落ち着き先が変わる様子を見てみましょう。

In [ ]:
def Lx(x):  return np.sin(3*x) + 0.3*x**2        # でこぼこした損失
def dLx(x): return 3*np.cos(3*x) + 0.6*x          # その微分 (勾配)

xs = np.linspace(-3.5, 3.5, 400)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xs, Lx(xs), color="gray", lw=1.5)

for x0, c in [(-3.0, "tab:red"), (-0.3, "tab:green"), (2.5, "tab:purple")]:
    x = x0
    for _ in range(200):
        x = x - 0.05 * dLx(x)                     # 勾配降下
    ax.scatter([x0], [Lx(x0)], color=c, marker="o", s=60, edgecolor="k", zorder=5,
               label=f"出発 x={x0:+.1f} → 到達 x={x:.2f}")
    ax.scatter([x], [Lx(x)], color=c, marker="*", s=220, edgecolor="k", zorder=5)
ax.set_xlabel("x (パラメータ)"); ax.set_ylabel("損失 L(x)"); ax.legend()
plt.show()

**観察**: 同じ勾配降下法でも、**出発点 (○) が違うと落ちる谷 (★) が変わり**、必ずしも最も深い谷にたどり着けません。これが**局所解の問題**です。

実際の学習では、これを和らげる工夫が使われます (講義の「学習における高度な話題」):
- **確率的勾配降下法 (SGD) / ミニバッチ**: 毎回データの一部だけで勾配を計算する。勾配が揺らぐので、浅い局所解や停滞から**抜け出しやすくなる**ことがある。
- **勾配消失問題への対策**: 層を深くすると、逆伝播で勾配が**どんどん小さくなって消え**、学習が止まることがある。これを防ぐ三大発明が **ReLU** (飽和しない活性化関数)・**スキップ接続** (ResNet)・**正規化層** (Batch Norm など)。

これらのおかげで、100 層を超える深いネットワークも学習できるようになりました。

---
## 演習 12-1

勾配降下法の更新式は $W \leftarrow W - \eta\,\nabla L(W)$ と、勾配に**マイナス**をつけて進みます。なぜマイナスなのか、**勾配の意味**から説明してください。

**解答例 (自分で考えてから開いてください)**

勾配 $\nabla L(W)$ は、定義から「$L$ が**最も急に増える方向**」を指すベクトル。実際、$W$ を $\Delta W$ だけ動かしたときの損失の変化は

$$
\Delta L \approx \nabla L \cdot \Delta W = |\nabla L|\,|\Delta W|\cos\theta
$$

で、勾配と同じ向き ($\cos\theta = 1$) のとき最も増える。**損失は減らしたい**のだから、その**真逆**($-\nabla L$ の向き) に進めばよい。だから更新式は勾配にマイナスをつける。$\eta$ はその 1 歩の大きさ (学習率)。

## 演習 12-2

線形モデル (第3回) では損失を微分して $=0$ と置けば**解析解**が得られたのに、ニューラルネットでは得られず、勾配降下法のような**数値計算**が必要になります。なぜでしょうか。

**解答例 (自分で考えてから開いてください)**

線形モデル $f(x)=w_0+w_1 x$ では二乗誤差 $L$ が $w$ の**2 次関数**になり、$\partial L/\partial w = 0$ が **$w$ について線形な連立方程式**になるので、きれいに解けて解析解が書ける。

一方ニューラルネットは、活性化関数を挟んだ**非線形な合成関数のオバケ**。損失 $L(W)$ は $W$ について複雑な非線形関数で、$\nabla L(W) = 0$ を満たす $W$ を**式で解くことができない**。そこで、勾配の情報を頼りに $W$ を少しずつ動かして $L$ を下げていく数値計算 (勾配降下法) に頼る。誤差逆伝播法は、その勾配を効率よく計算するための手段。

## 演習 12-3

§4 (回帰) または §5 (識別) のコードで、**学習率 (`lr`)・エポック数・活性化関数 (`nn.ReLU` ↔ `nn.Tanh`)** のいずれかを変えて実行し、損失の下がり方やフィット・決定境界がどう変わるか観察してください。

**解答例 (観察のヒント)**

- **学習率を小さく** (例 `lr=1e-4`) すると、同じエポック数では損失が十分下がりきらず、フィットが甘くなる (§2 の「小さすぎ→遅い」と同じ)。逆に**大きすぎる**と損失が暴れたり発散したりする。
- **エポック数を増やす**と損失はさらに下がるが、ある程度で頭打ちになる (これ以上下がらない)。
- **活性化関数を ReLU ↔ Tanh** で変えると、回帰では曲線の滑らかさ (ReLU は折れ線的、Tanh は滑らか)、識別では決定境界の形が変わる。いずれも「非線形だから曲げられる」点は共通。

`optim.Adam(model.parameters(), lr=...)` の `lr` を書き換えるのが手軽です。`optim.SGD(...)` に変えて素の勾配降下法と比べてみるのも面白いです。

---
# まとめ

- ニューラルネットの**学習**とは、**損失関数 $L(W)$ を最小化**するパラメータ $W$ を探すこと。線形モデルと違い**解析解は得られない**ので、数値計算で少しずつ $W$ を動かす。
- **勾配降下法** $W \leftarrow W - \eta\,\nabla L(W)$: 損失が最も急に増える方向 (勾配) の**逆**へ進む。**学習率 $\eta$** が小さすぎると遅く、大きすぎると発散する。
- **誤差逆伝播法**: 合成関数の微分 (連鎖律) を**出力側から伝播**させ、一度の計算で全層の勾配を得る効率的な方法。PyTorch では `loss.backward()` という**自動微分**として使える。
- PyTorch の学習ループは「**順伝播 → 損失 → `backward()` → `step()`**」の繰り返し。これで**曲線の回帰**も**線形分離できないデータの識別** (ドーナツ) もできた。
- 高度な話題: 勾配降下法は**局所解**に捕まりうる。**SGD/ミニバッチ**、**ReLU・スキップ接続・正規化層**などの工夫で、深いネットワークの学習が可能になった。

**次回の予告**: 次回 (第13回) は、画像で活躍する **CNN (畳み込みニューラルネット)** や、生成モデルの **VAE** など、**代表的なニューラルネットワーク**とその応用例を学びます。